In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd

from config import RAW_DATA_DIR, PROCESSED_DATA_DIR

In [2]:
icustays_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "icustays_clean.parquet",
    columns=["subject_id", "hadm_id", "stay_id", "intime", "outtime"]
)

icustays_df["intime"] = pd.to_datetime(icustays_df["intime"])
icustays_df["outtime"] = pd.to_datetime(icustays_df["outtime"])

print(icustays_df.shape)
print(icustays_df["stay_id"].duplicated().sum())

(94444, 5)
0


In [3]:
lab_item_map = {
    50862: "albumin",
    50863: "alkaline_phosphatase",
    50861: "alt",
    50868: "anion_gap",
    50878: "ast",
    50882: "bicarbonate",
    50885: "bilirubin_total",
    51006: "bun",
    50893: "calcium",
    50902: "chloride",
    50912: "creatinine",
    50931: "glucose",
    51221: "hematocrit",
    51222: "hemoglobin",
    51237: "inr",
    50813: "lactate",
    50960: "magnesium",
    50818: "pco2",
    50820: "ph",
    50970: "phosphate",
    51265: "platelets",
    50821: "po2",
    50971: "potassium",
    51274: "pt",
    51275: "ptt",
    50983: "sodium",
    51301: "wbc",
}

value_ranges = {
    "albumin": (0, 10),
    "alkaline_phosphatase": (0, 10000),
    "alt": (0, 20000),
    "anion_gap": (0, 100),
    "ast": (0, 20000),
    "bicarbonate": (0, 60),
    "bilirubin_total": (0, 80),
    "bun": (0, 300),
    "calcium": (0, 25),
    "chloride": (50, 150),
    "creatinine": (0, 50),
    "glucose": (0, 1500),
    "hematocrit": (0, 80),
    "hemoglobin": (0, 25),
    "inr": (0, 20),
    "lactate": (0, 30),
    "magnesium": (0, 20),
    "pco2": (0, 200),
    "ph": (6.5, 8.0),
    "phosphate": (0, 30),
    "platelets": (0, 3000),
    "po2": (0, 700),
    "potassium": (0, 10),
    "pt": (0, 150),
    "ptt": (0, 200),
    "sodium": (100, 180),
    "wbc": (0, 500),
}

selected_itemids = set(lab_item_map)

In [4]:
d_labitems_df = pd.read_csv(RAW_DATA_DIR / "d_labitems.csv")

selected_labitems_df = d_labitems_df[d_labitems_df["itemid"].isin(selected_itemids)][
    ["itemid", "label", "fluid", "category"]
].copy()

selected_labitems_df["lab_feature"] = selected_labitems_df["itemid"].map(lab_item_map)
selected_labitems_df.sort_values(["lab_feature", "itemid"])

,itemid,label,fluid,category,lab_feature
60,50862,Albumin,Blood,Chemistry,albumin
61,50863,Alkaline Phosphatase,Blood,Chemistry,alkaline_phosphatase
59,50861,Alanine Aminotransferase (ALT),Blood,Chemistry,alt
66,50868,Anion Gap,Blood,Chemistry,anion_gap
76,50878,Asparate Aminotransferase (AST),Blood,Chemistry,ast
80,50882,Bicarbonate,Blood,Chemistry,bicarbonate
83,50885,"Bilirubin, Total",Blood,Chemistry,bilirubin_total
202,51006,Urea Nitrogen,Blood,Chemistry,bun
91,50893,"Calcium, Total",Blood,Chemistry,calcium
100,50902,Chloride,Blood,Chemistry,chloride


In [5]:
usecols = [
    "labevent_id",
    "subject_id",
    "hadm_id",
    "itemid",
    "charttime",
    "valuenum",
    "valueuom",
    "flag",
    "priority",
]

chunksize = 1_000_000
cohort_hadm_ids = set(icustays_df["hadm_id"])
clean_chunks = []

for chunk in pd.read_csv(
    RAW_DATA_DIR / "labevents.csv.gz",
    usecols=usecols,
    chunksize=chunksize,
):
    chunk = chunk[
        chunk["hadm_id"].notna()
        & chunk["hadm_id"].isin(cohort_hadm_ids)
        & chunk["itemid"].isin(selected_itemids)
        & chunk["valuenum"].notna()
    ].copy()

    if chunk.empty:
        continue

    chunk["hadm_id"] = chunk["hadm_id"].astype("int64")
    chunk["charttime"] = pd.to_datetime(chunk["charttime"])
    chunk["lab_feature"] = chunk["itemid"].map(lab_item_map)

    chunk = chunk.merge(
        icustays_df[["subject_id", "hadm_id", "stay_id", "intime", "outtime"]],
        on=["subject_id", "hadm_id"],
        how="inner"
    )

    chunk = chunk[
        chunk["charttime"].between(chunk["intime"], chunk["outtime"], inclusive="both")
    ].copy()

    chunk["hours_from_icu_admit"] = (
        (chunk["charttime"] - chunk["intime"])
        .dt.total_seconds()
        .div(3600)
    )

    valid_range_mask = pd.Series(True, index=chunk.index)
    for feature, (lower, upper) in value_ranges.items():
        feature_mask = chunk["lab_feature"].eq(feature)
        valid_range_mask &= ~feature_mask | chunk["valuenum"].between(lower, upper, inclusive="both")

    chunk = chunk[valid_range_mask].copy()

    clean_chunks.append(
        chunk[
            [
                "labevent_id",
                "subject_id",
                "hadm_id",
                "stay_id",
                "charttime",
                "hours_from_icu_admit",
                "itemid",
                "lab_feature",
                "valuenum",
                "valueuom",
                "flag",
                "priority",
            ]
        ]
    )

if clean_chunks:
    labevents_clean_df = pd.concat(clean_chunks, ignore_index=True)
else:
    labevents_clean_df = pd.DataFrame(columns=[
        "labevent_id",
        "subject_id",
        "hadm_id",
        "stay_id",
        "charttime",
        "hours_from_icu_admit",
        "itemid",
        "lab_feature",
        "valuenum",
        "valueuom",
        "flag",
        "priority",
    ])

print(labevents_clean_df.shape)
print(labevents_clean_df.isna().sum())
print(labevents_clean_df["lab_feature"].value_counts())

(12140259, 12)
labevent_id                   0
subject_id                    0
hadm_id                       0
stay_id                       0
charttime                     0
hours_from_icu_admit          0
itemid                        0
lab_feature                   0
valuenum                      0
valueuom                 356421
flag                    6114408
priority                1879575
dtype: int64
lab_feature
potassium               612066
sodium                  610052
chloride                598694
hematocrit              577489
bicarbonate             575624
creatinine              574204
anion_gap               573044
bun                     572800
glucose                 562815
magnesium               559508
ph                      546288
phosphate               531419
calcium                 527227
hemoglobin              524367
platelets               523630
wbc                     514737
po2                     508835
pco2                    508351
ptt               

In [6]:
labevents_clean_df = labevents_clean_df.sort_values(
    ["stay_id", "lab_feature", "charttime"]
).reset_index(drop=True)

print(labevents_clean_df.duplicated().sum())
print(labevents_clean_df[["hours_from_icu_admit", "valuenum"]].describe())

0
       hours_from_icu_admit      valuenum
count          1.214026e+07  1.214026e+07
mean           1.225929e+02  5.212472e+01
std            2.101672e+02  1.736962e+02
min            0.000000e+00  0.000000e+00
25%            1.373000e+01  5.000000e+00
50%            4.890000e+01  1.600000e+01
75%            1.446697e+02  6.600000e+01
max            5.397826e+03  1.979000e+04


In [7]:
labevents_clean_df.to_parquet(
    PROCESSED_DATA_DIR / "labevents_clean.parquet",
    index=False
)